In [2]:
#Roger Fisher 2/17/2026
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Import CRUD module from Project One
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################

#Credentials for MongoDB
username = "aacuser"
password = "SNHU1234"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# Load the initial dataset
df = pd.DataFrame.from_records(db.read({}))

# MongoDB returns an _id ObjectId field that is not JSON-serialized for the Dash datatable
if "_id" in df.columns:
    df.drop(columns=['_id'],inplace=True)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load and encode the Grazioso Salvare Logo
image_filename = 'Grazioso_Salvare_Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read()).decode()

app.layout = html.Div([
    # Header
    html.Center(
    html.Div([
        html.H1("CS340 Dashboard", style={"marginBottom": "0px"}),

        html.P(
            "Roger Fisher • AAC Outcomes Explorer",
            style={
                "marginTop": "4px",
                "fontStyle": "italic",
                "opacity": "0.85"
            }
        ),

        # Clickable logo link (opens SNHU in a new tab)
        html.Div(
            html.A(
                html.Img(
                    src=f"data:image/png;base64,{encoded_image}",
                    style={
                        "height": "85px",
                        "marginTop": "12px",
                        "marginBottom": "8px"
                    }
                ),
                href="https://www.snhu.edu",
                target="_blank"
            )
        ),

        #The required unique identifier
        html.Div(
            "Unique Identifier: The Force is strong with this one",
            style={
                "display": "inline-block",
                "padding": "6px 14px",
                "border": "1px solid #ccc",
                "borderRadius": "20px",
                "backgroundColor": "#f7f7f7"
            }
        )
    ])
),
        
    
    # Interactive filter controls 
    html.Div([
        html.H3("Rescue Type Filter", style={"marginBottom": "6px"}),

        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain / Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster / Individual Tracking", "value": "disaster"},
                {"label": "Reset (All Records)", "value": "reset"},
            ],
            value="reset",   # default selection
            inline=True
        ),

        html.P(
            "Tip: Select a rescue type above to filter the table and charts.",
            style={"fontSize": "0.95em", "opacity": "0.85", "marginTop": "8px"}
        )
    ]),

    html.Hr(),
    
    # Interactive data table (supports sorting, filtering, pagination, and row selection)
    dash_table.DataTable(
        id='datatable-id',
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict('records'),
        
        page_size=10,
        sort_action="native",
        filter_action="native",
        
        # Single row selection is used to drive the map marker
        row_selectable="single",
        selected_rows=[0],
        
        # Styling for readability 
        style_header={"fontWeight": "bold"},
        style_table={"overflowX": "auto", "border": "1px solid #ddd"},
        style_cell={"padding": "6px", "fontFamily": "Arial", "textAlign": "left", "whiteSpace": "normal"},
    ),
    
    html.Br(),
    html.Hr(),
    
    # Side by side layout: pie chart on the left, and the map on the right
    html.Div(
    style={"display": "flex", "gap": "16px", "alignItems": "stretch"},
    children=[
        html.Div(
            id="graph-id",
            style={"flex": "1", "minWidth": "0"}  
        ),
        html.Div(
            id="map-id",
            style={"flex": "1", "minWidth": "0"}
        )
    ]
)
])

#############################################
# Interaction Between Components / Controller
#############################################


@app.callback(
    Output('datatable-id', 'data'),
    Output('datatable-id', 'selected_rows'),
    Input('filter-type', 'value')
)
def update_dashboard(filter_type):
# Updates the DataTable based on which rescue filter is selected 
# Each filter runs a MongoDB query through the CRUD module
    
    
    if filter_type == "water":
        query = {
            "animal_type": "Dog",
            "breed":{"$in":["Labrador Retriever Mix", "Chesapeake Bay Retriever", "Newfoundland"]},
            "sex_upon_outcome": "Intact Female",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "mountain":
        query = {
            "animal_type": "Dog",
            "breed":{"$in":["German Shepherd", "Alaskan Malamute", "Old English Sheepdog", "Siberian Husky", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156}
        }

    elif filter_type == "disaster":
        query = {
            "animal_type": "Dog",
            "breed":{"$in":["Doberman Pinscher", "German Shepherd", "Golden Retriever", "Bloodhound", "Rottweiler"]},
            "sex_upon_outcome": "Intact Male",
            "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300}
        }

    else:
        query = {}

    results = db.read(query)
    temp_df = pd.DataFrame.from_records(results)

    # Remove MongoDB _id field to avoid Dash serialization issues
    if "_id" in temp_df.columns:
        temp_df.drop(columns=["_id"], inplace=True)

    # Default row selection to first visible row for map stability
    return temp_df.to_dict("records"), [0]



@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):
    # Builds a pie chart showing the top 10 breeds currently visible
    if not viewData:
        return html.Div("No data to display.")

    dff = pd.DataFrame(viewData)

    # Count breeds and keep only the top 10 (prevents messy charts)
    breed_counts = (
        dff["breed"]
        .fillna("Unknown")
        .value_counts()
        .head(10)
        .reset_index()
    )
    breed_counts.columns = ["breed", "count"]

    fig = px.pie(
        breed_counts,
        names="breed",
        values="count",
        title="Top 10 Breeds (Current Filter)"
    )

    # Make it fit nicely in the left panel
    fig.update_layout(margin=dict(l=20, r=20, t=60, b=20))

    return dcc.Graph(figure=fig)
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    # Highlights selected columns in the DataTable to improve visibility
    if not selected_columns:
        selected_columns = []
    
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]



@app.callback(
    Output("map-id", "children"),
    [Input("datatable-id", "derived_virtual_data"),
     Input("datatable-id", "derived_virtual_selected_rows")]
)
def update_map(viewData, index):
    # Updates the map marker based on the selected row in the DataTable
    # If no row is selected or no data is visible, a default Austin area map is displayed

    # If table is empty or not ready, show blank Austin map
    if not viewData:
        return [
            dl.Map(
                style={"width": "100%", "height": "500px"},
                center=[30.75, -97.48],
                zoom=10,
                children=[dl.TileLayer(id="base-layer-id")]
            )
        ]

    dff = pd.DataFrame(viewData)

    # default to first row if none selected
    if not index:
        row = 0
    else:
        row = index[0]

    if row >= len(dff):
        row = 0

    # Use column names instead of numeric iloc indexes (safer)
    lat = dff.loc[row, "location_lat"]
    lon = dff.loc[row, "location_long"]
    breed = dff.loc[row, "breed"]
    name = dff.loc[row, "name"]
    outcome = dff.loc[row, "outcome_type"]

    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[lat, lon],  # centers on selected record (nice polish)
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(breed)),
                        dl.Popup([
                            html.H4(str(name)),
                            html.P(f"Breed: {breed}"),
                            html.P(f"Outcome: {outcome}")
                        ])
                    ]
                )
            ]
        )
    ]


# Starts the Dash server in the Codio Jupyter environment
app.run_server() 

Dash app running on https://virgoquiet-sharpliquid-3000.codio.io/proxy/8050/
